In [6]:
!pip install -U langchain langchain-community langchain-core langchain-huggingface chromadb faiss-cpu tiktoken wikipedia

# wikipedia retriever

In [7]:
from langchain_community.retrievers import WikipediaRetriever

In [10]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')

query = 'Geopolitics of India-Pakistan relations'

docs = retriever.invoke(query)

In [11]:
for i,doc in enumerate(docs):
  print(f'\n---result {i+1} ---')
  print(f'content: \n--{doc.page_content}.....')



---result 1 ---
content: 
--Iran and Pakistan established relations on 14 August 1947, the day of the independence of Pakistan, when Iran became the first country to recognize Pakistan. Both countries generally maintain a cordial relationship with formed alliances in a number of areas of mutual interest, such as combating the drug trade along their border and the cross-border insurgency in Balochistan. 
During the Cold War (1945–1991), both countries were part of the Western Bloc against the Eastern Bloc. They were founding members of the anti-communist alliance CENTO. Iran aided Pakistan in the India–Pakistan war of 1965 and India–Pakistan war of 1971, and backed Pakistan in the Bangladesh Liberation War and Indo-Pakistani War of 1971. Both countries shared a common animosity towards Baloch separatists and cooperated in the 1970s Balochistan operation. Following the Iranian Revolution of 1979, which overthrew the Pahlavi dynasty, Pakistan was one of the first countries to recognize t

# Vector Store Retriever

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [13]:
# Step 1: Your source documents
documents = [
Document (page_content="LangChain helps developers build LLM applications easily."),
Document (page_content="Chroma is a vector database optimized for LLM-based search."),
Document (page_content="Embeddings convert text into high-dimensional vectors."),
Document(page_content="OpenAI provides powerful embedding models.")
]

In [14]:
embeding_model = HuggingFaceEmbeddings()

vectorstore = Chroma.from_documents(documents=documents,
                     embedding = embeding_model,
                     collection_name= 'my_collection')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
retriever = vectorstore.as_retriever(search_kwargs={'k':2})

In [16]:
result = retriever.invoke(query)

In [17]:
for i,res in enumerate(result):
  print(f'\n---{i+1}---')
  print(f'content: \n{res.page_content}.....')


---1---
content: 
Embeddings convert text into high-dimensional vectors......

---2---
content: 
LangChain helps developers build LLM applications easily......


# MMR Retriever

In [18]:
from langchain_community.vectorstores import FAISS

In [19]:
# Sample documents
docs = [
Document(page_content="LangChain makes it easy to work with LLMs."),
Document (page_content="LangChain is used to build LLM based applications."),
Document (page_content="Chroma is used to store and search document embeddings."),
Document(page_content="Embeddings are vector representations of text."),
Document(page_content="MMR helps you get diverse results when doing similarity search."),
Document (page_content="LangChain supports Chroma, FAISS, Pinecone, and more.")
]

In [20]:
embeding_model = HuggingFaceEmbeddings()

vectorstore = FAISS.from_documents(embedding=embeding_model,
                                 documents=docs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
retriever = vectorstore.as_retriever(search_type='mmr',
                                     search_kwargs={'k':3, 'lambda_mult':0.5})

In [22]:
query ='what is langchain?'
resulrt = retriever.invoke(query)

In [23]:
for i,res in enumerate(result):
  print(f'\n-----{i+1}-----')
  print(f'content: \n{res.page_content}...')


-----1-----
content: 
Embeddings convert text into high-dimensional vectors....

-----2-----
content: 
LangChain helps developers build LLM applications easily....


# Multi query retriever(mqr)

In [30]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain.retrievers.multi_query import MultiQueryRetriever

ModuleNotFoundError: No module named 'langchain.retrievers'

In [31]:
# Relevant health & wellness documents
all_docs = [
Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
Document (page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
Document (page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
Document(page_content='The solar energy system in modern homes helps balance electricity demand.', metadata={"source": "I1"}),
Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [32]:
from chromadb.api.types import Embedding
embeding_model = HuggingFaceEmbeddings()

vectorstore = FAISS.add_documents(embedding=embeding_model, documents=all_docs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TypeError: VectorStore.add_documents() missing 1 required positional argument: 'self'

In [33]:
similarity = vectorstore.as_retriever(search_kwargs={'k':5}, search_type='similarity')

In [34]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    temperature=0.5
)

multiquery_retriver = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(search_kwargs={'k':5}),
                                                   llm = llm)

NameError: name 'MultiQueryRetriever' is not defined

In [35]:
query = 'how to improve energy level and maintain balance'

In [37]:
similarity_result = similarity.invoke(query)
multiquery_result = multiquery_retriver.invoke(query)

NameError: name 'multiquery_retriver' is not defined

In [38]:
for i, docs in enumerate(similarity_result):
  print(f'\n---{i+1}---')
  print(f'content--\n{docs.page_content}..')

print('*'*150)

for i, docs in enumerate(multiquery_result):
  print(f'\n---{i+1}---')
  print(f'content--\n{docs.page_content}..')


---1---
content--
LangChain makes it easy to work with LLMs...

---2---
content--
LangChain supports Chroma, FAISS, Pinecone, and more...

---3---
content--
LangChain is used to build LLM based applications...

---4---
content--
Embeddings are vector representations of text...

---5---
content--
MMR helps you get diverse results when doing similarity search...
******************************************************************************************************************************************************


NameError: name 'multiquery_result' is not defined

# contextual compression retrival

In [40]:
from langchain_community.vectorstores import FAISS
from langchain.retrievers. contextual_compression import ContextualCompressionRetriever
from langchain. retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

ModuleNotFoundError: No module named 'langchain.retrievers'

In [41]:
# Recreate the document objects from the previous data
docs = [
Document(page_content=(
"""The Grand Canyon is one of the most visited natural wonders in the world.
Photosynthesis is the process by which green plants convert sunlight into energy.
Millions of tourists travel to see it every year. The rocks date back millions of years."""
), metadata={"source": "Doc1"}),

Document (page_content=(
"""In medieval Europe, castles were built primarily for defense.
The chlorophyll in plant cells captures sunlight during photosynthesis.
Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
), metadata={"source": "Doc2"}),

Document(page_content=(
"""Basketball was invented by Dr. James Naismith in the late 19th century.
It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
), metadata={"source": "Doc3"}),
]

In [42]:
# create embeding midel and vector store

embedding_model = HuggingFaceEmbeddings()
vectorstore = FAISS.add_documents(embedding=embeding_model, documents=docs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TypeError: VectorStore.add_documents() missing 1 required positional argument: 'self'

In [43]:
base_retriever = vectorstore.as_retriever(search_kwargs={'k':2})

In [45]:
compressor = LLMChainExtractor.from_llm(llm)

NameError: name 'LLMChainExtractor' is not defined

In [46]:
compressor_retriever = ContextualCompressionRetriever(base_retriever=base_retriever,
                                        base_compressor = compressor)

NameError: name 'ContextualCompressionRetriever' is not defined

In [47]:
query = 'what is photosynthesis?'
compressor_result = compressor_retriever.invoke(query)

NameError: name 'compressor_retriever' is not defined

In [48]:
for i, docs in enumerate(compressor_result):
  print(f'\n---{i+1}---')
  print(f'content--\n{docs.page_content}..')

NameError: name 'compressor_result' is not defined